# EthnoAnatolia Cross-Ref
### Continuity of Anatolian medicinal plant uses from Dioscorides' *De materia medica* (1st c. CE) to present-day Turkish folk medicine (TEMDB)

**EthnoHack 2026 · Track 1 · Team botanists**

> ⚠️ **Disclaimer:** This notebook analyses *historical and traditional claims*. They do **not** establish efficacy or safety and are **not medical advice**.

**What this notebook does**
1. Loads the Dioscorides ↔ TEMDB cross-reference dataset (46 species, 606 Dioscorides use records, ICD-11 coded).
2. Runs data-quality checks.
3. Re-computes the continuity metrics from the category-level data and checks them against the stored values (reproducibility check).
4. Compares the expert-assigned continuity status with an objective, threshold-based class.
5. Produces the cross-reference table of **shared** ICD-11 categories, charts, and CSV outputs.

**How to run:** `Runtime → Run all`. No files need to be uploaded.

## 1. Setup

In [ ]:
import urllib.request
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_colwidth', 120)
pd.set_option('display.width', 200)

# GitHub location of the data file (used when the notebook runs in Colab)
GITHUB_USER = 'fatimabeyhan0332'
REPO = 'makeathon-2026-botanists-traditional-knowledge'
DATA_FILE = 'Dioscorides_TEMDB_Continuity.xlsx'

OUT = Path('outputs')
OUT.mkdir(exist_ok=True)

## 2. Load data
The workbook is read from the repository's `data/` folder (locally) or downloaded from GitHub (in Colab).

In [ ]:
candidates = [Path('../data') / DATA_FILE, Path('data') / DATA_FILE, Path(DATA_FILE)]
path = next((p for p in candidates if p.exists()), None)
if path is None:
    url = f'https://raw.githubusercontent.com/{GITHUB_USER}/{REPO}/main/data/{DATA_FILE}'
    print('Downloading', url)
    urllib.request.urlretrieve(url, DATA_FILE)
    path = Path(DATA_FILE)
print('Using', path)

xls = pd.ExcelFile(path)
print('Sheets:', xls.sheet_names)

uses = pd.read_excel(xls, 'Dioscorides_Uses')       # one row = one plant x one Dioscorides use
plants = pd.read_excel(xls, 'Plant_Summary')        # one row = one plant
detail = pd.read_excel(xls, 'Disease_Detail')       # one row = one plant x one ICD-11 category
stored = pd.read_excel(xls, 'Disease_Summary')      # stored plant-level metrics

for df in (uses, plants, detail, stored):
    df['Plant_Name'] = df['Plant_Name'].astype(str).str.strip()

# Keep only rows that belong to the 46 study species (drops the summary rows at the bottom of Disease_Summary)
species = set(plants['Plant_Name'])
stored = stored[stored['Plant_Name'].isin(species)].copy()

plants['Continuity_Status'] = plants['Continuity_Status'].astype(str).str.strip()
detail['Status'] = detail['Status'].astype(str).str.strip()
detail['TEMDB_Records'] = pd.to_numeric(detail['TEMDB_Records'], errors='coerce').fillna(0)
detail['Dioscorides_Use_Count'] = pd.to_numeric(detail['Dioscorides_Use_Count'], errors='coerce').fillna(0)

print(f"{len(species)} species · {len(uses)} Dioscorides use records · {len(detail)} plant x ICD-11 category rows")

## 3. Data overview and quality checks

In [ ]:
# 3a. Same species in every sheet?
for name, df in [('Dioscorides_Uses', uses), ('Disease_Detail', detail), ('Disease_Summary', stored)]:
    s = set(df['Plant_Name'])
    print(f'{name:18s} species={len(s):3d}  missing={sorted(species - s)}  extra={sorted(s - species)}')

# 3b. Expected label values
print('Plant status values   :', sorted(plants['Continuity_Status'].unique()))
print('Category status values:', sorted(detail['Status'].unique()))

# 3c. Source edition used for each Dioscorides use
src = uses['Dioscorides_Source'].astype(str)
edition = pd.Series('Other', index=uses.index)
edition[src.str.contains('Sprengel')] = 'Sprengel 1829 (Latin)'
edition[src.str.contains('Wellmann')] = 'Wellmann 1906 (Greek)'
print()
print(edition.value_counts().to_string())

# 3d. ICD-11 coding status of the Dioscorides uses
code = uses['ICD11_Code'].fillna('').astype(str).str.strip()
chapter = uses['ICD11_Chapter'].fillna('').astype(str)
is_adverse = chapter.str.startswith('Adverse effect')   # harmful effects reported by Dioscorides (not coded)
is_unclassified = chapter.str.contains('Unclassified')   # therapeutic effects without a disease (diuretic, ...)
is_coded = ~is_adverse & ~is_unclassified
valid = code.str.match(r'^[0-9A-Z]{4}([.][0-9A-Z]+)?$') | code.str.match(r'^[0-9A-Z]{4}.[0-9A-Z]{4}$')
print()
print(f'Coded with an ICD-11 code      : {is_coded.sum()}')
print(f'Unclassified therapeutic effect: {is_unclassified.sum()}')
print(f'Adverse effect (not coded)     : {is_adverse.sum()}')
print(f'Coded rows with unexpected code format: {(is_coded & ~valid).sum()}')
if (is_coded & ~valid).any():
    display(uses.loc[is_coded & ~valid, ['Plant_Name', 'Dioscorides_Use', 'ICD11_Code', 'ICD11_Title']])

In [ ]:
# ICD-11 chapters of the Dioscorides uses (coded rows only)
top_ch = uses.loc[is_coded, 'ICD11_Chapter'].value_counts().head(12)
ax = top_ch.sort_values().plot.barh(figsize=(9, 5), color='#4C72B0')
ax.set_xlabel('Number of Dioscorides use records')
ax.set_title('ICD-11 chapters of uses described by Dioscorides (top 12)')
plt.tight_layout(); plt.savefig(OUT / 'dioscorides_icd11_chapters.png', dpi=150); plt.show()

## 4. Continuity metrics (re-computed from category-level data)

For each species, every ICD-11 category in which the plant is used is labelled **Shared** (used in both sources), **Dioscorides only**, or **TEMDB only** (the latter split by whether a Dioscorides use falls in the same ICD-11 block or chapter).

| Metric | Definition |
|---|---|
| TEMDB coverage – category (%) | share of TEMDB records whose ICD-11 category also appears in Dioscorides |
| TEMDB coverage – block / chapter (%) | same, cumulative, at ICD-11 block / chapter level |
| Dioscorides retention (%) | share of Dioscorides therapeutic uses whose category is still found in TEMDB (adverse effects excluded) |
| Jaccard | shared categories / all categories |

All categories (including *only* rows) are needed as denominators; the cross-reference table in Section 6 then shows only the **shared** categories.

In [ ]:
d = detail.copy()
d['shared'] = d['Status'].eq('Shared')
d['to_block'] = d['Status'].isin(['Shared', 'TEMDB only (same ICD-11 block)'])
d['to_chapter'] = d['Status'].isin(['Shared', 'TEMDB only (same ICD-11 block)', 'TEMDB only (same ICD-11 chapter)'])
d['temdb_cat'] = d['TEMDB_Records'].where(d['shared'], 0)
d['temdb_block'] = d['TEMDB_Records'].where(d['to_block'], 0)
d['temdb_chapter'] = d['TEMDB_Records'].where(d['to_chapter'], 0)
d['dio_shared'] = d['Dioscorides_Use_Count'].where(d['shared'], 0)

g = d.groupby('Plant_Name')
m = pd.DataFrame({
    'TEMDB_records': g['TEMDB_Records'].sum(),
    'categories_total': g.size(),
    'categories_shared': g['shared'].sum(),
    'temdb_cat': g['temdb_cat'].sum(),
    'temdb_block': g['temdb_block'].sum(),
    'temdb_chapter': g['temdb_chapter'].sum(),
    'dio_shared': g['dio_shared'].sum(),
})
# denominator = therapeutic uses only (adverse effects reported by Dioscorides are excluded)
m['Dioscorides_uses'] = uses[~is_adverse].groupby('Plant_Name').size()
m['coverage_category_pct'] = 100 * m['temdb_cat'] / m['TEMDB_records']
m['coverage_block_pct'] = 100 * m['temdb_block'] / m['TEMDB_records']
m['coverage_chapter_pct'] = 100 * m['temdb_chapter'] / m['TEMDB_records']
m['dioscorides_retention_pct'] = 100 * m['dio_shared'] / m['Dioscorides_uses']
m['jaccard'] = m['categories_shared'] / m['categories_total']

def objective_class(p):
    # thresholds are arbitrary and used only for comparison with the expert status
    if p >= 50: return 'High'
    if p >= 20: return 'Medium'
    if p > 0: return 'Low'
    return 'None'

m['objective_class'] = m['coverage_category_pct'].apply(objective_class)
m = m.join(plants.set_index('Plant_Name')[['Turkish_Local_Names', 'Continuity_Status']])

# Species whose historical identification is debated in the literature
DISPUTED = ('Rosa canina', 'Petroselinum crispum', 'Rosmarinus officinalis', 'Salvia rosmarinus')
m['identification_disputed'] = [any(n.startswith(x) for x in DISPUTED) for n in m.index]

m = m.sort_values('coverage_category_pct', ascending=False)
cols = ['Turkish_Local_Names', 'TEMDB_records', 'Dioscorides_uses', 'categories_shared', 'categories_total',
        'coverage_category_pct', 'coverage_block_pct', 'coverage_chapter_pct',
        'dioscorides_retention_pct', 'jaccard', 'objective_class', 'Continuity_Status', 'identification_disputed']
display(m[cols].round(1))

### 4a. Reproducibility check
The re-computed metrics are compared with the values stored in the workbook (sheet `Disease_Summary`). Differences ≤ 0.1 are rounding.

In [ ]:
pairs = {
    'TEMDB_records': 'TEMDB_Records',
    'Dioscorides_uses': 'Dioscorides_Uses',
    'coverage_category_pct': 'TEMDB_Coverage_Category_Pct',
    'coverage_block_pct': 'TEMDB_Coverage_Block_Pct',
    'coverage_chapter_pct': 'TEMDB_Coverage_Chapter_Pct',
    'dioscorides_retention_pct': 'Dioscorides_Retention_Pct',
    'categories_shared': 'Shared_Categories',
    'categories_total': 'Total_Categories',
    'jaccard': 'Jaccard',
}
s = stored.set_index('Plant_Name')
rows, differing = [], {}
for new, old in pairs.items():
    if old not in s.columns:
        rows.append((new, 'column not found', None)); continue
    diff = (m[new] - pd.to_numeric(s[old], errors='coerce')).abs()
    tol = 0.01 if new == 'jaccard' else 0.1
    bad = diff[diff > tol]
    if len(bad): differing[new] = list(bad.index)
    rows.append((new, f'{len(bad)} of {diff.notna().sum()} species differ', round(diff.max(), 3)))
check = pd.DataFrame(rows, columns=['metric', 'result', 'max_abs_difference'])
display(check)
for k, v in differing.items():
    print(f'{k}: {v}')

### 4b. Overall continuity (record-weighted, all 46 species)

In [ ]:
tot = m[['TEMDB_records', 'temdb_cat', 'temdb_block', 'temdb_chapter', 'dio_shared', 'Dioscorides_uses']].sum()
overall = pd.Series({
    'TEMDB records in a shared ICD-11 category': 100 * tot.temdb_cat / tot.TEMDB_records,
    '... in the same ICD-11 block': 100 * tot.temdb_block / tot.TEMDB_records,
    '... in the same ICD-11 chapter': 100 * tot.temdb_chapter / tot.TEMDB_records,
    'Dioscorides uses still found in TEMDB': 100 * tot.dio_shared / tot.Dioscorides_uses,
}).round(1)
print(f'TEMDB records: {int(tot.TEMDB_records)} · Dioscorides therapeutic uses: {int(tot.Dioscorides_uses)}')
print(overall.astype(str).add(' %').to_string())

ax = overall.iloc[:3].plot.bar(figsize=(7, 4), color=['#2a9d8f', '#8ab17d', '#e9c46a'], rot=0)
ax.set_xticklabels(['Category', 'Block', 'Chapter'])
ax.set_ylabel('% of TEMDB records')
ax.set_ylim(0, 100)
ax.set_title('How much of today\'s use was already described by Dioscorides?\n(cumulative ICD-11 levels)')
for p in ax.patches:
    ax.annotate(f'{p.get_height():.1f}%', (p.get_x() + p.get_width() / 2, p.get_height() + 1.5), ha='center')
plt.tight_layout(); plt.savefig(OUT / 'overall_continuity.png', dpi=150); plt.show()

## 5. Species-level continuity: expert status vs objective class

In [ ]:
COLORS = {'Full agreement': '#2a9d8f', 'Partial agreement': '#e9c46a', 'Different use': '#e76f51'}
p = m.sort_values('coverage_category_pct')
labels = [f'{n}  *' if dis else n for n, dis in zip(p.index, p['identification_disputed'])]
fig, ax = plt.subplots(figsize=(9, 13))
ax.barh(labels, p['coverage_category_pct'], color=[COLORS.get(s, 'grey') for s in p['Continuity_Status']])
ax.set_xlabel('% of TEMDB records in an ICD-11 category also described by Dioscorides')
ax.set_title('Continuity per species\n(bar colour = expert status; * = disputed historical identification)')
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=c, label=s) for s, c in COLORS.items()], loc='lower right', title='Expert status')
plt.tight_layout(); plt.savefig(OUT / 'species_continuity.png', dpi=150); plt.show()

print('Expert status (rows) vs objective class (columns):')
display(pd.crosstab(m['Continuity_Status'], m['objective_class'], margins=True)
        .reindex(columns=['High', 'Medium', 'Low', 'None', 'All'], fill_value=0))

## 6. Cross-reference table — shared ICD-11 categories only
Each row is a disease category for which the same species is used both by Dioscorides and in present-day Anatolian folk medicine (TEMDB). Dioscorides uses are English translations of the team's summaries of the Latin/Greek text.

In [ ]:
xref = (d.loc[d['shared'], ['Plant_Name', 'ICD11_Category', 'ICD11_Chapter', 'TEMDB_Records', 'TEMDB_Uses',
                            'Dioscorides_Use_Count', 'Dioscorides_Uses']]
          .sort_values(['Plant_Name', 'TEMDB_Records'], ascending=[True, False])
          .reset_index(drop=True))
xref[['TEMDB_Records', 'Dioscorides_Use_Count']] = xref[['TEMDB_Records', 'Dioscorides_Use_Count']].astype(int)
print(f'{len(xref)} shared plant x ICD-11 category pairs across {xref.Plant_Name.nunique()} species')
display(xref)

# Which ICD-11 chapters show the strongest continuity?
ch = xref.groupby('ICD11_Chapter').agg(shared_pairs=('Plant_Name', 'size'), species=('Plant_Name', 'nunique'))
display(ch.sort_values('shared_pairs', ascending=False))

## 7. Save outputs

In [ ]:
m[cols].round(2).to_csv(OUT / 'species_continuity_metrics.csv', encoding='utf-8-sig')
xref.to_csv(OUT / 'crossref_shared_categories.csv', index=False, encoding='utf-8-sig')
overall.to_csv(OUT / 'overall_continuity.csv', header=['percent'], encoding='utf-8-sig')
print('Saved:', sorted(f.name for f in OUT.iterdir()))

try:  # offer downloads when running in Google Colab
    from google.colab import files
    import shutil
    shutil.make_archive('ethnoanatolia_outputs', 'zip', OUT)
    files.download('ethnoanatolia_outputs.zip')
except ImportError:
    pass

## 8. Limitations
- **Coverage of TEMDB:** TEMDB currently covers Turkish ethnobotanical literature from 2018–2020. A use that is absent from TEMDB is *not* evidence that it has disappeared from Anatolian folk medicine.
- **TEMDB version:** values come from the team's corrected, not-yet-published revision of the TEMDB Zenodo record (DOI 10.5281/zenodo.20002971).
- **Historical identification:** species marked with * (e.g. *kynosbatos* → *Rosa canina*) have debated identifications.
- **ICD-11 mapping of ancient terms** involves interpretation (e.g. quartan fever → 1F42); therapeutic effects without a disease (diuretic, emmenagogue, …) are kept as *Unclassified*.
- **Translation:** Dioscorides uses were first summarised in Turkish from the Latin/Greek text and then translated into English; ancient terms are kept in parentheses (e.g. *aigilops*, *phyma*).
- **Chapter numbers** follow Sprengel's numbering; OCR may introduce ±1 deviations.
- The objective-class thresholds (50 / 20 %) are arbitrary and used only for comparison with the expert assessment.

> Historical claims are not treatments. Nothing in this notebook should be used to prepare or take a remedy.